# CholecT45 Instrument Annotations - Exploration & Loading

**Purpose**: Load and explore the instrument annotations from CholecT45 dataset

**Dataset Info**:
- 45 cholecystectomy videos (subset of Cholec80)
- 6 instrument types annotated
- Frame-by-frame binary labels (0=not present, 1=present)
- 1 FPS sampling rate

In [ ]:
# Cell 1: Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json

print("✅ Imports complete")

In [ ]:
# Cell 2: Setup Paths
CHOLECT45_DIR = Path("CholecT45")  # Update if different

# Check if dataset exists
if not CHOLECT45_DIR.exists():
    print(f"❌ Dataset not found at: {CHOLECT45_DIR}")
    print("\nPlease update the path or extract the dataset")
else:
    print(f"✅ Dataset found at: {CHOLECT45_DIR}")
    
    # Show structure
    subdirs = [d.name for d in CHOLECT45_DIR.iterdir() if d.is_dir()]
    print(f"\nSubdirectories: {subdirs}")

In [ ]:
# Cell 3: Load Instrument Mapping
instrument_dict_path = CHOLECT45_DIR / "dict" / "instrument.txt"

if instrument_dict_path.exists():
    # Read instrument names
    with open(instrument_dict_path, 'r') as f:
        instrument_lines = f.readlines()
    
    # Parse instrument mapping
    INSTRUMENT_MAPPING = {}
    for line in instrument_lines:
        if line.strip():
            parts = line.strip().split()
            if len(parts) >= 2:
                idx = int(parts[0])
                name = ' '.join(parts[1:])
                INSTRUMENT_MAPPING[idx] = name
    
    print("Instrument Mapping:")
    print("=" * 50)
    for idx, name in sorted(INSTRUMENT_MAPPING.items()):
        print(f"  ID {idx}: {name}")
    
    print(f"\n✅ Found {len(INSTRUMENT_MAPPING)} instrument types")
else:
    print(f"❌ Instrument dictionary not found at: {instrument_dict_path}")

In [ ]:
# Cell 4: List Available Videos
instrument_dir = CHOLECT45_DIR / "instrument"
video_files = sorted(instrument_dir.glob("VID*.txt"))

print(f"Found {len(video_files)} videos with instrument annotations:")
print("=" * 50)
for vid_file in video_files[:10]:
    print(f"  - {vid_file.name}")
if len(video_files) > 10:
    print(f"  ... and {len(video_files) - 10} more")

In [ ]:
# Cell 5: Load Sample Instrument Annotations (VID01)
sample_video = "VID01"
sample_path = instrument_dir / f"{sample_video}.txt"

# Load annotations (tab-separated or space-separated)
# Column 0: Frame index
# Columns 1-6: Binary labels for 6 instruments
df = pd.read_csv(sample_path, sep='\t', header=None)

# If tab separation didn't work, try space separation
if df.shape[1] == 1:
    df = pd.read_csv(sample_path, sep=' ', header=None)

print(f"Loaded annotations for {sample_video}")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"Columns: {df.shape[1]} (frame index + 6 instruments)")
print(f"Total frames: {len(df)}")
print("\nFirst 10 rows:")
print(df.head(10))

In [ ]:
# Cell 6: Rename Columns
column_names = ['frame_idx'] + [INSTRUMENT_MAPPING.get(i, f'instrument_{i}') for i in range(6)]
df.columns = column_names

print(f"Renamed columns for {sample_video}:")
print("=" * 50)
print(df.head(10))
print("\nColumn names:", df.columns.tolist())

In [ ]:
# Cell 7: Analyze Instrument Frequency
instrument_cols = [col for col in df.columns if col != 'frame_idx']

print(f"Instrument Frequency Analysis for {sample_video}")
print("=" * 50)
print(f"Total frames: {len(df)}\n")

for instrument in instrument_cols:
    count = df[instrument].sum()
    percentage = 100 * count / len(df)
    print(f"{instrument:20s}: {count:5d} frames ({percentage:5.1f}%)")

# Frames with at least one instrument
frames_with_instruments = (df[instrument_cols].sum(axis=1) > 0).sum()
print(f"\n{'Frames with instruments':20s}: {frames_with_instruments:5d} frames ({100*frames_with_instruments/len(df):5.1f}%)")

In [ ]:
# Cell 8: Visualize Instrument Presence Over Time
fig, axes = plt.subplots(len(instrument_cols) + 1, 1, figsize=(15, 10))

# Plot each instrument
for idx, instrument in enumerate(instrument_cols):
    axes[idx].fill_between(df['frame_idx'], 0, df[instrument], alpha=0.6)
    axes[idx].set_ylabel(instrument, fontsize=10)
    axes[idx].set_ylim(-0.1, 1.1)
    axes[idx].set_yticks([0, 1])
    axes[idx].grid(True, alpha=0.3)
    if idx < len(instrument_cols) - 1:
        axes[idx].set_xticks([])

# Plot total instruments per frame
total_instruments = df[instrument_cols].sum(axis=1)
axes[-1].plot(df['frame_idx'], total_instruments, linewidth=1, color='black')
axes[-1].fill_between(df['frame_idx'], 0, total_instruments, alpha=0.3, color='gray')
axes[-1].set_ylabel('Total\nInstruments', fontsize=10)
axes[-1].set_xlabel('Frame Index', fontsize=12)
axes[-1].grid(True, alpha=0.3)

plt.suptitle(f'Instrument Presence Over Time - {sample_video}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cholect45_instrument_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualization saved: cholect45_instrument_timeline.png")

In [ ]:
# Cell 9: Co-occurrence Analysis
print("Instrument Co-occurrence Matrix")
print("=" * 50)
print("How often instruments appear together:\n")

# Calculate co-occurrence
cooccurrence = df[instrument_cols].T.dot(df[instrument_cols])

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cooccurrence, cmap='YlOrRd', aspect='auto')

# Labels
ax.set_xticks(range(len(instrument_cols)))
ax.set_yticks(range(len(instrument_cols)))
ax.set_xticklabels(instrument_cols, rotation=45, ha='right')
ax.set_yticklabels(instrument_cols)

# Add values
for i in range(len(instrument_cols)):
    for j in range(len(instrument_cols)):
        text = ax.text(j, i, int(cooccurrence.iloc[i, j]),
                      ha="center", va="center", color="black", fontsize=10)

plt.colorbar(im, ax=ax, label='Co-occurrence Count')
plt.title(f'Instrument Co-occurrence - {sample_video}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cholect45_cooccurrence.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualization saved: cholect45_cooccurrence.png")

In [ ]:
# Cell 10: Create Helper Function to Load Any Video
def load_instrument_annotations(video_id, cholect45_dir=CHOLECT45_DIR):
    """
    Load instrument annotations for a given video
    
    Args:
        video_id: Video ID (e.g., 'VID01', 'VID02', or just 1, 2)
        cholect45_dir: Path to CholecT45 dataset
    
    Returns:
        DataFrame with columns: frame_idx, grasper, bipolar, hook, scissors, clipper, irrigator
    """
    # Handle both 'VID01' and 1 as input
    if isinstance(video_id, int):
        video_id = f"VID{video_id:02d}"
    
    # Load annotations
    annotation_path = cholect45_dir / "instrument" / f"{video_id}.txt"
    
    if not annotation_path.exists():
        raise FileNotFoundError(f"Annotation file not found: {annotation_path}")
    
    # Try tab-separated first, then space-separated
    df = pd.read_csv(annotation_path, sep='\t', header=None)
    if df.shape[1] == 1:
        df = pd.read_csv(annotation_path, sep=' ', header=None)
    
    # Rename columns
    column_names = ['frame_idx'] + [INSTRUMENT_MAPPING.get(i, f'instrument_{i}') for i in range(6)]
    df.columns = column_names
    
    return df

# Test the function
test_df = load_instrument_annotations('VID01')
print(f"✅ Helper function created")
print(f"\nTest: Loaded VID01 with {len(test_df)} frames")
print(test_df.head())

In [ ]:
# Cell 11: Load All Videos and Create Summary Statistics
print("Loading all videos for summary statistics...\n")

all_video_stats = []

for vid_file in video_files:
    video_id = vid_file.stem
    
    # Load annotations
    df_vid = load_instrument_annotations(video_id)
    
    # Calculate statistics
    instrument_cols = [col for col in df_vid.columns if col != 'frame_idx']
    
    stats = {
        'video_id': video_id,
        'total_frames': len(df_vid),
        'frames_with_instruments': (df_vid[instrument_cols].sum(axis=1) > 0).sum(),
    }
    
    # Per-instrument statistics
    for instrument in instrument_cols:
        stats[f'{instrument}_frames'] = df_vid[instrument].sum()
    
    all_video_stats.append(stats)

# Create summary DataFrame
summary_df = pd.DataFrame(all_video_stats)

print("Summary Statistics for All Videos:")
print("=" * 80)
print(summary_df.head(10))

# Save to CSV
summary_df.to_csv('cholect45_instrument_summary.csv', index=False)
print(f"\n✅ Summary saved: cholect45_instrument_summary.csv")

In [ ]:
# Cell 12: Overall Dataset Statistics
print("Overall CholecT45 Instrument Statistics")
print("=" * 80)

print(f"\nTotal videos: {len(summary_df)}")
print(f"Total frames: {summary_df['total_frames'].sum():,}")
print(f"Frames with instruments: {summary_df['frames_with_instruments'].sum():,} ({100*summary_df['frames_with_instruments'].sum()/summary_df['total_frames'].sum():.1f}%)")

print("\nInstrument Distribution Across All Videos:")
print("-" * 80)
for instrument in instrument_cols:
    total_frames = summary_df[f'{instrument}_frames'].sum()
    percentage = 100 * total_frames / summary_df['total_frames'].sum()
    num_videos = (summary_df[f'{instrument}_frames'] > 0).sum()
    print(f"{instrument:20s}: {total_frames:6,} frames ({percentage:5.1f}%) in {num_videos:2d}/{len(summary_df)} videos")

In [ ]:
# Cell 13: Check Video ID Mapping to Cholec80
print("Checking CholecT45 to Cholec80 Video Mapping")
print("=" * 80)
print("\nCholecT45 videos (VID01-VID45) are a subset of Cholec80 (video01-video80)")
print("\nAssuming direct mapping:")
print("  VID01 → video01 (used for blood segmentation training)")
print("  VID02 → video02 (used for blood temporal tracking)")
print("  VID03 → video03 (used for blood temporal tracking)")
print("\nNOTE: Verify this mapping by checking frame counts!")
print("\nExpected frame counts (from your blood tracking):")
print("  video01: 1,734 frames (at 1 FPS)")
print("  video02: 2,840 frames (at 1 FPS)")
print("  video03: 5,829 frames (at 1 FPS)")

print("\nActual frame counts from CholecT45:")
for vid_id in ['VID01', 'VID02', 'VID03']:
    if vid_id in summary_df['video_id'].values:
        count = summary_df[summary_df['video_id'] == vid_id]['total_frames'].values[0]
        print(f"  {vid_id}: {count:,} frames")
    else:
        print(f"  {vid_id}: NOT FOUND")

print("\n✅ If frame counts match, mapping is correct!")

In [ ]:
# Cell 14: Export Helper Functions for Next Notebook
# Save key information for use in bleeding prediction pipeline

export_data = {
    'instrument_mapping': INSTRUMENT_MAPPING,
    'instrument_names': list(INSTRUMENT_MAPPING.values()),
    'num_instruments': len(INSTRUMENT_MAPPING),
    'dataset_path': str(CHOLECT45_DIR),
    'total_videos': len(summary_df),
    'total_frames': int(summary_df['total_frames'].sum()),
}

with open('cholect45_config.json', 'w') as f:
    json.dump(export_data, f, indent=2)

print("✅ Configuration exported: cholect45_config.json")
print("\nExported data:")
print(json.dumps(export_data, indent=2))

## Summary

### What We Discovered:
1. CholecT45 contains 45 videos with 6 instrument types annotated
2. Frame-level binary labels (0/1) for each instrument
3. Instruments: Grasper, Bipolar, Hook, Scissors, Clipper, Irrigator
4. Direct mapping to Cholec80 videos (VID01 = video01, etc.)

### Next Steps:
1. ✅ Verified instrument annotations are available
2. ✅ Created helper function to load any video
3. ✅ Analyzed instrument distribution and co-occurrence
4. **Next**: Combine with blood segmentation features for bleeding prediction

### Files Generated:
- `cholect45_instrument_timeline.png` - Temporal visualization
- `cholect45_cooccurrence.png` - Co-occurrence matrix
- `cholect45_instrument_summary.csv` - Summary statistics
- `cholect45_config.json` - Configuration for next notebook